# 🐍 Master Guide: JSON, CSV & Data Serialization
> **Single-Source Revision Guide**: Covers structured data formats, custom JSON encoders, streaming CSV, and the security perils of `pickle`.
>
> *Run each cell interactively with `Shift + Enter`.*

---

## 📌 1. Topic Overview
Serialization converts in-memory Python objects into byte/text streams for storage or network transport. JSON is standard for web APIs, CSV for tabular data, and Pickle for Python-specific binary serialization.

---
## 🧠 2. CPython Architecture: Serialization Security & Benchmarks

| Format | Speed | Human-Readable | Cross-Language | Secure Against Arbitrary Execution? |
| :--- | :--- | :--- | :--- | :--- |
| **JSON** | Fast (C-accelerated) | Yes | Yes (Universal) | **Yes** |
| **CSV** | Very Fast | Yes | Yes | **Yes** |
| **Pickle** | Fast (Binary) | No | No (Python only) | **NO! (Remote Code Execution)** |

### Why `pickle` is Dangerous (Arbitrary Code Execution)
`pickle` is a stack-based virtual machine. When unpickling, it can execute the `__reduce__` method of any class, allowing attackers to run arbitrary shell commands (`os.system`)! **Never unpickle data from untrusted sources.**

---
## 🟢 3. Part 1: Beginner Fundamentals

### Step 1: JSON Parsing & Dumping (`json.dumps`, `json.loads`)
- `json.dump(obj, f)` / `json.dumps(obj)`: Serialize object to file / string.
- `json.load(f)` / `json.loads(s)`: Deserialize file / string to object.

In [ ]:
import json

data = {
    "user": "Alice",
    "role": "Admin",
    "active": True,
    "permissions": ["read", "write", "delete"]
}

# Serialize to formatted JSON string:
json_str = json.dumps(data, indent=2)
print("Serialized JSON string:")
print(json_str)

# Deserialize back to Python dict:
parsed = json.loads(json_str)
print("Parsed user:", parsed["user"])

### Step 2: Tabular Data with `csv.DictReader` and `csv.DictWriter`

In [ ]:
import csv

csv_filename = "employees.csv"
fieldnames = ["id", "name", "department", "salary"]

# Writing with csv.DictWriter:
with open(csv_filename, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerow({"id": 1, "name": "Alex", "department": "Engineering", "salary": 95000})
    writer.writerow({"id": 2, "name": "Beth", "department": "Marketing", "salary": 82000})

# Reading with csv.DictReader:
with open(csv_filename, "r", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(f"{row['name']} works in {row['department']} (${row['salary']})")

---
## 🟡 4. Part 2: Intermediate Features

### Custom JSON Encoders (`default` parameter)
JSON does not natively support `datetime`, `Decimal`, or custom classes. Use `default` to handle them!

In [ ]:
from datetime import datetime

class User:
    def __init__(self, name, registered_at):
        self.name = name
        self.registered_at = registered_at

user = User("Sarah", datetime(2026, 10, 25, 12, 0, 0))

# Custom serializer function:
def custom_serializer(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    if isinstance(obj, User):
        return {"name": obj.name, "registered_at": obj.registered_at}
    raise TypeError(f"Object of type {type(obj)} is not JSON serializable")

user_json = json.dumps(user, default=custom_serializer, indent=2)
print("Custom JSON output:")
print(user_json)

---
## 🔴 5. Part 3: Tricky Parts, Gotchas & Interview Traps

### ⚠️ Gotcha 1: CSV Extra Blank Lines Trap (`newline=""`)
On Windows, opening CSV files without `newline=""` results in **double newlines (`\r\r\n`)** between rows!

In [ ]:
print("Rule: When working with csv.writer or csv.reader, ALWAYS specify open(..., newline='')!")

### ⚠️ Gotcha 2: The Pickle Security Remote Code Execution Trap
`pickle.loads()` can execute arbitrary operating system commands during deserialization!

In [ ]:
import pickle

class ExploitPayload:
    def __reduce__(self):
        # The return tuple specifies a callable and its arguments
        return (print, ("EXPLOIT TRIGGERED: Arbitrary code executed on unpickle!",))

payload = pickle.dumps(ExploitPayload())
print("Simulating unpickle of untrusted payload:")
_ = pickle.loads(payload)
print("Never use pickle for user-supplied data; use JSON or Protocol Buffers!")

---
## 📋 6. Part 4: Quick Revision Cheat Sheet

```
+------------------------------------+---------------------------------------+
| Concept                            | Key Rule                              |
+------------------------------------+---------------------------------------+
| json.dump / json.load              | Writes to / reads from file stream    |
| json.dumps / json.loads            | Writes to / reads from string         |
| open(csv_file, newline='')         | newline='' avoids blank lines on Win  |
| Custom JSON encoder                | Pass default=callable to json.dumps   |
| Pickle Security                    | NEVER unpickle untrusted data (RCE!)  |
+------------------------------------+---------------------------------------+
```

---
## 🎓 Practical Hands-On & Real-World Examples

### 1. CSV Reading & Writing (`CsvDictReaderEx1.py`, `CsvDictWriteEx1.py`)
Using `csv.DictReader` and `csv.DictWriter` for header-aware structured record processing.

In [ ]:
# Practical Implementation: CSV Processing with DictReader / DictWriter
import csv
import io

# 1. Writing structured student data to CSV buffer (CsvDictWriteEx1.py)
output_buffer = io.StringIO()
fieldnames = ["sno", "sname", "marks", "grade"]
writer = csv.DictWriter(output_buffer, fieldnames=fieldnames)
writer.writeheader()

students_data = [
    {"sno": 101, "sname": "Ananya", "marks": 94.5, "grade": "Distinction"},
    {"sno": 102, "sname": "Bharath", "marks": 81.0, "grade": "First Class"},
    {"sno": 103, "sname": "Chaitra", "marks": 67.5, "grade": "Second Class"}
]
for s in students_data:
    writer.writerow(s)

csv_content = output_buffer.getvalue()
print("Generated CSV Content:")
print(csv_content)

# 2. Reading back with csv.DictReader (CsvDictReaderEx1.py)
input_buffer = io.StringIO(csv_content)
reader = csv.DictReader(input_buffer)
print("Parsed Records:")
for row in reader:
    print(f"  Student #{row['sno']:<4} | Name: {row['sname']:<10} | Marks: {row['marks']:<5} | Grade: {row['grade']}")

### 2. JSON & Object Pickling Serialization (`DictDataToJsonFile.py`, `StudPickEx1.py`)
Converting Python dictionaries to JSON and serializing complex objects with `pickle`.

In [ ]:
# JSON & Pickle Serialization Drill
import json
import pickle

course_catalog = {
    "institute": "Python Technologies",
    "course": "Python Full Stack",
    "modules": ["Basics", "OOPs", "File IO", "Threading", "DB"],
    "fees": 12000,
    "active": True
}

# 1. JSON String serialization (dumps & loads)
json_str = json.dumps(course_catalog, indent=2)
print("Serialized JSON:\n", json_str)
parsed_dict = json.loads(json_str)
print("Deserialized keys:", list(parsed_dict.keys()))

# 2. Binary Object Pickling (pickle.dumps & loads)
class EmployeeRecord:
    def __init__(self, emp_id, name, salary):
        self.emp_id = emp_id
        self.name = name
        self.salary = salary

emp = EmployeeRecord(501, "Divya", 92000.0)
pickled_bytes = pickle.dumps(emp)
print(f"\nPickled Employee Object: {len(pickled_bytes)} bytes")

restored_emp = pickle.loads(pickled_bytes)
print(f"Restored Object via Pickle: ID={restored_emp.emp_id}, Name={restored_emp.name}, Sal={restored_emp.salary}")